In [ ]:
#!uv pip uninstall sentencepiece protobuf

In [2]:
import os
from dotenv import load_dotenv

# Load key-value pairs from the .env file into the system environment
load_dotenv(override=True)

# Safely extract variables using os.getenv()
api_key = os.getenv("HF_TOKEN")
#print("HF_TOKEN loaded from .env:", api_key)

In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import LlamaTokenizer

model_id = "Nashxi/bankaccountagreement-tinyllama-domain-lora-merged"

# FIX 1: Load tokenizer from the base model since the merged repo lacks tokenizer files
tokenizer = AutoTokenizer.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T",
    use_fast=False
)

# Load your custom merged model weights
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    token=api_key,
    torch_dtype=torch.bfloat16, 
    device_map="auto"
)

# Your domain-specific prompt
prompt = "If your account is a type listed under “Personal Accounts” in our product information, can i use for business purposes?"

# FIX 2: Generate tokens first so we can check their shape safely
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# Debug prints (Corrected syntax)
print("Input IDs:", inputs["input_ids"]) 
print("Shape:", inputs["input_ids"].shape)

# Run text generation safely without tensor shape errors
outputs = model.generate(
    **inputs, 
    max_new_tokens=512, 
    do_sample=True, 
    temperature=0.7
)

print("\nGenerated Output:")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Loading weights: 100%|██████████| 201/201 [00:11<00:00, 17.23it/s]
Some parameters are on the meta device because they were offloaded to the disk.


Input IDs: 

[transformers] Both `max_new_tokens` (=512) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


tensor([[    1,   960,   596,  3633,   338,   263,  1134,  9904,  1090,  1346,
          7435,   284, 16535, 29879, 30024,   297,  1749,  3234,  2472, 29892,
           508,   474,   671,   363,  5381, 11976, 29973]], device='mps:0')
Shape: torch.Size([1, 27])

Generated Output:
If your account is a type listed under “Personal Accounts” in our product information, can i use for business purposes? Yes. You may use your Chase Personal Checking, Chase Business Checking, or Chase Interest Arbitration Checking account to make arbitration claims. We do not have to accept your arbitration request if your account is a type listed under “Personal Accounts” in our product information. How do I make a claim for reimbursement? To make a claim, you must file a “RiverRs Claims Arbitration Aggregate Claim” within sixty (60) days after the day we confirm that we sent you a final offer of resolution (generally within seven days after we send you the Offer to Resolve). For personal accounts, the last da

In [10]:
from ast import If

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import LlamaTokenizer
import textwrap

model_id = "Nashxi/bankaccountagreement-tinyllama-domain-lora-instruction"

# FIX 1: Load tokenizer from the base model since the merged repo lacks tokenizer files
tokenizer = AutoTokenizer.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T",
    use_fast=False
)

# Load your custom merged model weights
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    token=api_key,
    torch_dtype=torch.bfloat16, 
    device_map="auto"
)

# Your domain-specific prompt
prompt = (
"you are bank account agreement expert. Answer me this question. "
"How many types of depositaccounts are there in the bank account agreement? "
"Do not answer in any other way. Only answer the question. "
"If you do not know the answer, say 'I don't know'."
"Do not ask me any questions. Do not answer in any other way. Only answer the question. " \
"Do not send the promt back to me. Do not answer in any other way. Only answer the question. "
)


# FIX 2: Generate tokens first so we can check their shape safely
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# Debug prints (Corrected syntax)
print("Input IDs:", inputs["input_ids"]) 
print("Shape:", inputs["input_ids"].shape)

# Run text generation safely without tensor shape errors
outputs = model.generate(
    **inputs, 
    max_new_tokens=512, 
    do_sample=True, 
    temperature=0.7
)
completion = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("\nGenerated Output:")
print(completion)
print(textwrap.fill(completion, width=80))

Loading weights: 100%|██████████| 308/308 [00:00<00:00, 4100.93it/s]
[transformers] Both `max_new_tokens` (=512) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Input IDs: tensor([[    1,   366,   526,  9124,  3633, 17327, 17924, 29889,   673,   592,
           445,  1139, 29889,  1128,  1784,  4072,   310, 19754,   277, 10149,
         29879,   526,   727,   297,   278,  9124,  3633, 17327, 29973,  1938,
           451,  1234,   297,   738,   916,   982, 29889,  9333,  1234,   278,
          1139, 29889,   960,   366,   437,   451,  1073,   278,  1234, 29892,
          1827,   525, 29902,  1016, 29915, 29873,  1073,  4286,  6132,   451,
          2244,   592,   738,  5155, 29889,  1938,   451,  1234,   297,   738,
           916,   982, 29889,  9333,  1234,   278,  1139, 29889,  1938,   451,
          3638,   278,  2504, 29873,  1250,   304,   592, 29889,  1938,   451,
          1234,   297,   738,   916,   982, 29889,  9333,  1234,   278,  1139,
         29889, 29871]], device='mps:0')
Shape: torch.Size([1, 102])

Generated Output:
you are bank account agreement expert. Answer me this question. How many types of depositaccounts are there in 